In [1]:
import numpy as np
import pandas as pd
import hf_hydrodata as hf


def get_ma2025_wtd(df, lon_col="longitude", lat_col="latitude"):
    df = df.copy()

    # Initialize output columns
    df["WTD_ma2025_m"] = np.nan
    df["WTD_ma2025_uncertainty_m"] = np.nan

    # Loop through sites
    for idx, row in df.iterrows():

        lat = row[lat_col]
        lon = row[lon_col]

        options_wtd = {
            "dataset": "ma_2025",
            "variable": "water_table_depth",
            "grid": "conus2_wtd.30",
            "latlon_point": [lat, lon],
        }

        wtd = hf.get_gridded_data(options_wtd).squeeze()
        df.loc[idx, "WTD_ma2025_m"] = float(wtd)

        options_unc = {
            "dataset": "ma_2025",
            "variable": "wtd_uncertainty",
            "grid": "conus2_wtd.30",
            "latlon_point": [lat, lon],
        }

        uncertainty = hf.get_gridded_data(options_unc).squeeze()
        df.loc[idx, "WTD_ma2025_uncertainty_m"] = float(uncertainty)

    return df

In [2]:
import pandas as pd

hf.register_api_pin("zhaoting@ualberta.ca", "0616")

traveltime_meta = pd.read_csv("traveltime_meta_table.csv")
test = pd.read_csv("test_table.csv")
test = test.rename(columns = {'id': 'test_id', 'test_latitude': 'latitude', 'test_longitude': 'longitude'})

df = traveltime_meta.merge(test[['test_id', 'latitude', 'longitude']], on='test_id', how = 'left')

# convert lat/lon to float
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

In [3]:
import numpy as np
grid = 'conus2_wtd.30'
lat = df.latitude.values
lon = df.longitude.values
dgwt = np.empty(len(lat), dtype=float)
for i in range(len(lat)):
    (x, y) = hf.from_latlon(grid, lat[i], lon[i])
    options = {
        'dataset':'ma_2025',
        'variable': 'water_table_depth',
        'grid': grid,
        'grid_point': [int(x), int(y)],
    }
    dgwt[i] = hf.gridded.get_gridded_data(options)[0][0]

df['dgwt_m'] = dgwt

In [9]:
# extract a dataframe for locations without dgwt_m
df_no_dgwt = df[df["dgwt_m"].isna()].reset_index(drop=True)

In [5]:
df.describe()

,id,test_id,horizontal_offset,travel_time_comment,latitude,longitude,dgwt_m
count,1302.000000,1302.000000,1302.000000,0.0,1302.000000,1302.000000,1155.000000
mean,651.500000,5618.705837,0.925416,NaN,36.401112,-104.478622,12.732630
std,375.999335,634.555140,0.119949,NaN,3.940577,15.910438,22.493944
min,1.000000,4856.000000,0.509016,NaN,25.947300,-122.477000,0.000000
25%,326.250000,5182.250000,0.960000,NaN,34.126275,-121.887350,3.049500
50%,651.500000,5511.500000,0.960000,NaN,36.041670,-97.545200,4.841890
75%,976.750000,5850.750000,0.960000,NaN,37.786500,-90.074725,9.485376
max,1302.000000,7525.000000,0.960000,NaN,47.179500,-79.467600,149.352005


In [10]:
df_no_dgwt.describe()

,id,test_id,horizontal_offset,travel_time_comment,latitude,longitude,dgwt_m
count,147.000000,147.000000,1.470000e+02,0.0,147.000000,147.000000,0.0
mean,428.891156,5289.659864,9.600000e-01,NaN,35.940965,-104.957841,NaN
std,397.037318,404.654656,1.114019e-16,NaN,3.250034,19.620432,NaN
min,15.000000,4870.000000,9.600000e-01,NaN,28.750000,-122.477000,NaN
25%,78.500000,4933.500000,9.600000e-01,NaN,32.789800,-122.252400,NaN
50%,214.000000,5070.000000,9.600000e-01,NaN,35.109800,-120.625000,NaN
75%,780.500000,5647.500000,9.600000e-01,NaN,37.786900,-80.146900,NaN
max,1201.000000,6075.000000,9.600000e-01,NaN,47.166400,-79.736400,NaN


In [11]:
df_no_dgwt

,id,test_id,travel_time_name,travel_time_method,travel_time_date,horizontal_offset,travel_time_comment,latitude,longitude,dgwt_m
0,15,4870,ALC001,SCPT,2000-11-07,0.96,NaN,37.7847,-122.2530,NaN
1,16,4871,ALC002,SCPT,2000-11-07,0.96,NaN,37.7802,-122.2463,NaN
2,17,4872,ALC003,SCPT,2000-11-07,0.96,NaN,37.7828,-122.2491,NaN
3,18,4873,ALC004,SCPT,2000-11-21,0.96,NaN,37.7829,-122.2478,NaN
4,19,4874,ALC005,SCPT,2000-11-15,0.96,NaN,37.7816,-122.2506,NaN
...,...,...,...,...,...,...,...,...,...,...
142,1057,5931,SOC031,SCPT,2004-03-04,0.96,NaN,35.1094,-120.6280,NaN
143,1058,5932,SOC032,SCPT,2004-03-04,0.96,NaN,35.1097,-120.6264,NaN
144,1059,5933,SOC033,SCPT,2004-03-04,0.96,NaN,35.1073,-120.6267,NaN
145,1060,5934,SOC034,SCPT,2004-03-04,0.96,NaN,35.1082,-120.6279,NaN


In [8]:
df.to_csv("ma_et_al_query.csv", index=False)

In [12]:
import pandas as pd
import requests
import time

df_no_dgwt

elevations = []

for i, row in df_no_dgwt.iterrows():
    lon = float(row["longitude"])
    lat = float(row["latitude"])
    
    url = "https://epqs.nationalmap.gov/v1/json"
    params = {
        "x": lon,
        "y": lat,
        "units": "Meters",
        "format": "json"
    }
    
    elevation = None
    for attempt in range(5):
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            elevation = float(response.json()["value"])
            print(f"Row {i}: ({lon}, {lat}) → {elevation}m")
            break

        except requests.exceptions.HTTPError as e:
            print(f"Row {i} attempt {attempt + 1}/5 failed: {e} — retrying in 10s")
            time.sleep(10)

        except (KeyError, ValueError) as e:
            print(f"Row {i}: parsing error at ({lon}, {lat}): {e}")
            break

    elevations.append(elevation)

df_no_dgwt["USGS_elevation_m"] = elevations


Row 0 attempt 1/5 failed: 504 Server Error: Gateway Timeout for url: https://epqs.nationalmap.gov/v1/json?x=-122.253&y=37.7847&units=Meters&format=json — retrying in 10s
Row 0: parsing error at (-122.253, 37.7847): Expecting value: line 1 column 1 (char 0)
Row 1 attempt 1/5 failed: 504 Server Error: Gateway Timeout for url: https://epqs.nationalmap.gov/v1/json?x=-122.2463&y=37.7802&units=Meters&format=json — retrying in 10s
Row 1: (-122.2463, 37.7802) → 4.809196949m
Row 2 attempt 1/5 failed: 504 Server Error: Gateway Timeout for url: https://epqs.nationalmap.gov/v1/json?x=-122.2491&y=37.7828&units=Meters&format=json — retrying in 10s


KeyboardInterrupt: 

In [2]:
gwt_Ma_et_al = pd.read_csv('ma_et_al_query.csv')
gwt_USGS = pd.read_csv('usgs_elevation_query.csv')

In [5]:
gwt_Ma_et_al

,id,test_id,travel_time_name,travel_time_method,travel_time_date,horizontal_offset,travel_time_comment,latitude,longitude,dgwt_m
0,1,4856,AGR002,SCPT,2004-09-08,0.960000,NaN,46.43000,-86.84530,3.350000
1,2,4857,AGR003,SCPT,2004-09-08,0.960000,NaN,46.42490,-86.84540,4.570000
2,3,4858,AGR005,SCPT,2004-09-08,0.960000,NaN,46.41430,-86.82620,7.921200
3,4,4859,AGR006,SCPT,2004-09-08,0.960000,NaN,46.41680,-86.83050,6.048759
4,5,4860,AGR007,SCPT,2004-09-09,0.960000,NaN,46.42950,-86.87210,4.298840
...,...,...,...,...,...,...,...,...,...,...
1297,1298,7521,SCPT-283,SCPT,2021-10-19,0.509016,NaN,36.37032,-120.25998,125.577599
1298,1299,7522,SCPT-200,SCPT,2021-10-26,0.509016,NaN,36.48201,-120.32083,126.492004
1299,1300,7523,SCPT-312,SCPT,2021-11-02,0.509016,NaN,36.45511,-120.30480,131.216400
1300,1301,7524,SCPT-147,SCPT,2021-11-04,0.509016,NaN,36.53659,-120.38861,134.112000


In [7]:
gwt_USGS.rename(columns={'USGS_elevation_m': 'USGS_gwt_m'}, inplace=True)

In [ ]:
gwt = pd.merge(gwt_USGS[['id', 'USGS_gwt_m']], gwt_Ma_et_al[['id', 'dgwt_m']], on='id', how='outer')


In [15]:
gwt["dgwt_m"] = gwt["dgwt_m"].combine_first(gwt["USGS_gwt_m"])

In [18]:
gwt.drop(columns=['USGS_gwt_m'], inplace=True)

In [23]:
gwt.describe()

,id,dgwt_m
count,1302.000000,1302.000000
mean,651.500000,12.113357
std,375.999335,22.552589
min,1.000000,0.000000
25%,326.250000,2.938780
50%,651.500000,4.572000
75%,976.750000,8.653593
max,1302.000000,184.730728


In [24]:
gwt.to_csv("gwt_master_table.csv", index=False)